### Notes
If the error message comes back as "SessionNotCreatedException: Message: session not created: 
This version of OperaDriver only supports Opera version xx",
get the latest Opera webdriver from https://www.selenium.dev/downloads/ or
https://github.com/operasoftware/operachromiumdriver/releases
and then save it here: C:/Users/hilton.netta/SeleniumDrivers/

In [128]:
# load libraries
import pandas as pd
import numpy as np
import os
import csv
import datetime
from datetime import date
from selenium import webdriver
from selenium.webdriver.opera.options import Options # to state absolute path of the Opera Browser is installed
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
#import networkdays as networkdays

In [129]:
# set paths to the driver, urls, and report paramters
os.environ["PATH"] = r"C:/Users/hilton.netta/SeleniumDrivers" # + os.pathsep + os.getenv("PATH")
# https://stackoverflow.com/questions/61213005/modify-beginning-of-path-variable-with-os-environ-in-python
        
url_default        = "https://pimportalprod2015.eagleaccess.com/Default.aspx"
url_derv           = "https://pimportalprod2015.eagleaccess.com/Queries/Query.aspx?rpt=DerivativeExposure"
url_holdings       = "https://pimportalprod2015.eagleaccess.com/Queries/Query.aspx?rpt=PortfolioAnalytics"

pth                = r'P:\Investment Operations\GRC\Compliance\Daily\derv.xlsx'

In [130]:
# set variables
derv_xlsx = pd.read_excel(pth, sheet_name = 'Report', header = None, index_col = 0, usecols = 'A:B', nrows = 4)
#fnds_       = ", ".join(pd.read_excel(pth, sheet_name = 'Report', usecols = 'D')['Derivative Checker Funds'].values.tolist())
#fnds_       = pd.read_excel(pth, sheet_name = 'Report', usecols = 'D')['Derivative Checker Funds'].values
fnds_       = ['PMMF', 'PABS', 'PIMBAL', 'GMRETF']

aladdin     = derv_xlsx.iat[0,0]
sesame      = derv_xlsx.iat[1,0]
#when       = derv_xlsx.iat[2,0]
when        = datetime.datetime(2022, 4, 17, 0, 0)
day_        = f'{when:%#d}' # f'{when:%d}' report date with leading zeroes
month_year_ = f'{when:%B}, {when:%Y}'
month_      = f'{when:%b}'
year_       = f'{when:%Y}'

print(aladdin, when, day_, month_, year_, month_year_, derv_xlsx.shape, fnds_)

hnetta 2022-04-17 00:00:00 17 Apr 2022 April, 2022 (4, 1) ['PMMF', 'PABS', 'PIMBAL', 'GMRETF']


In [131]:
fnds_

['PMMF', 'PABS', 'PIMBAL', 'GMRETF']

In [132]:
# open a browser on the web page
driver = webdriver.Chrome()
driver.get(url_derv) # Derivative Exposure page

# set a maximum waiting period for elements to become available
wait = WebDriverWait(driver, 10) # https://selenium-python.readthedocs.io/waits.html

# login with credentials
driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_UserName').send_keys(aladdin)
driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_Password').send_keys(sesame)
driver.find_element(By.CSS_SELECTOR, '#LoginCtrl_MainLoginControl_LoginButton').click()

# having logged in, switch to the query page
driver.find_element(By.CSS_SELECTOR, '#ModifyLinkLabel').click()

edit_criteria_window = driver.window_handles[0] # save curent window handle
# How to switch to new window in Selenium for Python?
# https://stackoverflow.com/questions/10629815/how-to-switch-to-new-window-in-selenium-for-python

In [136]:
# get the web elements of fund list, and of the submit button
fund_selector  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedIds"]')

# assign values to the fund list elements
driver.execute_script(f'arguments[0].value = "{fnds_}";', fund_selector)

In [134]:
# open the calendar
cal_popup       = driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_B-1"]')
cal_popup.click()

# get the comma separator between month and date. Avoids the default 'month year' without a separating comma
wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_NMC"]'))).click() # (right) shift calendar to next month
wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_PMC"]'))).click() #  (left) shift calendar to previous month

# set up the calendar left-clicker
lmonth_selector = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_PMC"]')))

In [140]:
# open the calendar
cal_popup       = driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_B-1"]')
cal_popup.click()

In [150]:
# click the month, year selector
month_year_selector = driver.find_element(By.CSS_SELECTOR, 'td[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_TC"]')
month_year_selector.click()

In [ ]:
calendar_shift

In [147]:
pre_calendar = driver.window_handles[0]
pre_calendar

'CDwindow-36413F99C1830500578BA410D86D33B1'

In [149]:
curr_calendar = driver.window_handles[0]
curr_calendar

'CDwindow-36413F99C1830500578BA410D86D33B1'

In [92]:
driver.find_element(By.XPATH,'//td[@class="dxeCalendarFastNavYear dxeCalendarFastNavYearSelected"]').text

'2021'

In [91]:
driver.find_element(By.XPATH,'//td[@class="dxeCalendarFastNavYearSelected"]').text

''

In [137]:
list(driver.find_elements(By.XPATH,'//td[@class="dxeCalendarFastNavYear"] | td[@class="dxeCalendarFastNavYear dxeCalendarFastNavYearSelected"]'))

[<selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="8baf7dd1-7824-4aa7-a4da-cd497b64cbb7")>,
 <selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="a3509a48-aacb-4ca2-b8e1-5ac75504583c")>,
 <selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="6c20faf0-3a25-4c3c-a88f-059be89785d0")>,
 <selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="d2c82d03-121e-45f7-a74d-63ec8c5d335d")>,
 <selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="4640ac7d-5475-4443-8b0a-de280dd64ffc")>,
 <selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="ee5002ce-5d7a-46bf-969c-3eb6280e1fe3")>,
 <selenium.webdriver.remote.webelement.WebElement (session="830ea84a8134c1fa84ee63b00e95f99b", element="cd5ea15d-8c98-4d92-a146-14

In [138]:
list(e.text for e in driver.find_elements(By.XPATH,'//td[@class="dxeCalendarFastNavYear"] | td[@class="dxeCalendarFastNavYear dxeCalendarFastNavYearSelected"] | td[@class="dxeCalendarFastNavYear dxeCalendarFastNavYearSelected dxeCalendarFastNavYearHover"]'))

['', '2040', '2041', '2042', '2043', '2045', '2046', '2047', '2048', '2049']

In [68]:
x = driver.find_elements(By.XPATH,'//td[@class="dxeCalendarFastNavYear"] | td[@class="dxeCalendarFastNavYear dxeCalendarFastNavYearSelected"]')
x

[<selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="53a1ce23-adcb-4b6b-a43c-939438c621b7")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="b49d6973-7bf2-4927-9d19-c8392268cf0b")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="a73e850f-8e06-4b4d-a0b5-8c5ce2427326")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="8769ac46-1366-48a8-8d05-409a1e979074")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="8838cedb-c640-4d59-844f-71995e90e7cf")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="61f156ff-7b83-4cfd-b96d-860af0bbd01f")>,
 <selenium.webdriver.remote.webelement.WebElement (session="c23b38463d09c8f911d51f4ebe449f29", element="b10bd051-1711-4f87-ae37-6e

In [66]:
# find the year shift-left clicker button
lyear_selector = driver.find_element(By.CSS_SELECTOR, 'img[class="dxEditors_edtCalendarFNPrevYear"]') # year shift-left selector

# find the correct year
while year_ not in list(e.text for e in driver.find_elements(By.XPATH,'//td[@class="dxeCalendarFastNavYear"] | td[@class="dxeCalendarFastNavYear dxeCalendarFastNavYearSelected"]')):
    lyear_selector.click()
    
# click the correct year

ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=108.0.5359.98)


In [123]:
month_

'Apr'

In [121]:
mo_ =  driver.find_elements(By.XPATH,f'//td[@class="dxeCalendarFastNavMonth"] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected"] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected dxeCalendarFastNavMonthHover"]')
for e in mo_:
    print(e.text, type(e.text))

 <class 'str'>
Jan <class 'str'>
Feb <class 'str'>
Mar <class 'str'>
Apr <class 'str'>
May <class 'str'>
Jun <class 'str'>
Jul <class 'str'>
Aug <class 'str'>
Sep <class 'str'>
Oct <class 'str'>
Nov <class 'str'>
Dec <class 'str'>


In [124]:
month_selector = driver.find_elements(By.XPATH,f'//td[@class="dxeCalendarFastNavMonth"][text()={month_}] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected"][text()={month_}]')
for e in month_selector:
    print(e)

In [ ]:
month_selector = driver.find_elements(By.XPATH,f'//td[@class="dxeCalendarFastNavMonth"][text()="{month_}"] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected"][text()="{month_}"] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected dxeCalendarFastNavMonthHover"][text()="{month_}"]')

In [127]:
# select the reporting month on the calendar
month_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarFastNavMonth"][text()="{month_}"] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected"][text()="{month_}"] | //td[@class="dxeCalendarFastNavMonth dxeCalendarFastNavMonthSelected dxeCalendarFastNavMonthHover"][text()="{month_}"]')
month_selector.click()

ElementNotInteractableException: Message: element not interactable
  (Session info: chrome=108.0.5359.98)


In [8]:
# left click the calendar until the reporting'month year' combination presents
while driver.find_element(By.XPATH,'//td[@id="ctl00_c_qc_QueryInputs_QueryInputsPopup_DATE1_DateCtrl_From_DDD_C_TC"]').text != month_year_:
    lmonth_selector.click()
# use Python Selenium to get span text; https://stackoverflow.com/questions/14590341/use-python-selenium-to-get-span-text
    
# select the reporting day on the calendar
day_selector = driver.find_element(By.XPATH,f'//td[@class="dxeCalendarDay"][text()={day_}] | //td[@class="dxeCalendarDay dxeCalendarWeekend"][text()={day_}]')
day_selector.click()

In [9]:
# click the 'Submit' button
submit_button  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_RunBtn"]')
submit_button.click()

In [70]:
fnds_

['PMMF', 'PABS', 'PIMBAL', 'GMRETF']

In [69]:
# enter fund names
name_      = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_FilterValue"]')
#name_tick_ = driver.find_element(By.CSS_SELECTOR, 'span[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SimpleFilterGrid_DXSelBtn0_D"]')
#name_tick_ = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SimpleFilterGrid_DXSelBtn0_D"]'))).click()

#name_.send_keys('LAUBAl')
#name_tick_.click()

for name in fnds_:
    name_.send_keys(name)
    #driver.execute_script(name, name_)
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'span[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SimpleFilterGrid_DXSelBtn0_D"]'))).click()
    name_.clear()

StaleElementReferenceException: Message: stale element reference: element is not attached to the page document
  (Session info: chrome=108.0.5359.95)


In [77]:
for handle in driver.window_handles:
    driver.switch_to.window(handle)
    print(driver.title)

Eagle Investment Systems | My Queries


In [10]:
x = driver.getWindowHandle[0]
x

AttributeError: 'WebDriver' object has no attribute 'getWindowHandle'

In [12]:
window_before = driver.window_handles[0]
window_before

'CDwindow-FAEA62D3378AC7ECB146DAB0B853444E'

In [83]:
# get the web elements of fund list, and of the submit button
fund_selector  = driver.find_element(By.CSS_SELECTOR, 'input[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedIds"]')

# assign values to the fund list elements
driver.execute_script(f'arguments[0].value = "IJGIPF, QIFFGIF";', fund_selector)

In [84]:
fund_selector.get_attribute("value")

'IJGIPF, QIFFGIF'

In [93]:
driver.find_element(By.CSS_SELECTOR, 'tr[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedItemsGrid_DXSyncInput"][dxo.keys]').text

InvalidSelectorException: Message: invalid selector: An invalid or illegal selector was specified
  (Session info: chrome=108.0.5359.72)


In [105]:
driver.find_element(By.XPATH, "//*[@id='dxss_1044391392']//*[dxo.keys=['IJGIPF','PMMF','QIFFGIF']]")

#//*[@id="dxss_1044391392"]/text()

InvalidSelectorException: Message: invalid selector: Unable to locate an element with the xpath expression //*[@id='dxss_1044391392']//*[dxo.keys=['IJGIPF','PMMF','QIFFGIF']] because of the following error:
SyntaxError: Failed to execute 'evaluate' on 'Document': The string '//*[@id='dxss_1044391392']//*[dxo.keys=['IJGIPF','PMMF','QIFFGIF']]' is not a valid XPath expression.
  (Session info: chrome=108.0.5359.72)


In [85]:
# check that funds list is accurate
len(fnds_), len(fund_selector.get_attribute("value"))

(788, 15)

In [ ]:
#fx = driver.find_element(By.CSS_SELECTOR, 'tr[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedItemsGrid_DXDataRow3"]')
fx = driver.find_element(By.CSS_SELECTOR, 'tr[id="ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedItemsGrid_DXSyncInput"]')


# assign values to the fund list elements
driver.execute_script(f'arguments[0].value = "IJGIPF, QIFFGIF";', fund_selector)
#ctl00_c_qc_QueryInputs_QueryInputsPopup_FUND0_SelectedItemsGrid_DXDataRow0